# Test Different Data
Workflow multi-agentico con **Pydantic `with_structured_output()`** + **generazione PDF sintetico**.
Modello: `google/gemma-4-31b-it`

In [45]:
import os, sys, json
os.chdir("/workspaces/hackaton-UNIBG-repo")
sys.path.insert(0, "src")
from dotenv import load_dotenv; load_dotenv()
print("API Key:", "OK" if os.getenv("OPEN_ROUTER_KEY") else "MANCA")

API Key: OK


## Modelli Pydantic

In [46]:
from typing import Any, Dict, List
from pydantic import BaseModel, Field

class FieldDefinition(BaseModel):
    field: str = Field(description="Nome del campo")
    type: str = Field(description="Tipo: string, number, date, list, object")
    sensitive: bool = Field(description="True se contiene dati personali")
    constraints: str = Field(description="Vincoli del campo")

class DocumentExtraction(BaseModel):
    schema_def: List[FieldDefinition] = Field(description="Schema del documento")
    data: Dict[str, Any] = Field(description="Dati estratti dal documento")

class ValidatedRecords(BaseModel):
    records: List[Dict[str, Any]] = Field(description="Record validati e corretti")

print("Modelli Pydantic definiti")

Modelli Pydantic definiti


## 1. OCR del PDF

In [47]:
from inference_different_data import extract_text_from_pdf

raw_text = extract_text_from_pdf("dataset/LABORATORIO_ANALISI_CLINICHE.pdf")
print(raw_text[:1200])

2026-05-29 08:50:53.657 | INFO     | inference_different_data:extract_text_from_pdf:49 - Estrazione OCR da: dataset/LABORATORIO_ANALISI_CLINICHE.pdf


--- Page 1 ---
LABORATORIO ANALISI CLINICHE "VITA NOVA"

DATI PAZIENTE

Nome e Cognome: Mario Rossi
Codice Fiscale: RSSMRA80A01H501U
Data di Nascita: 01/01/1980 |

Indirizzo: Via Roma 1, 00100 Roma

INFORMAZIONI PRELIEVO

Data Prelievo: 29/05/2026

Medico Richiedente: Dott.ssa Giulia Bianchi

REFERTO DI LABORATORIO - CHIMICA CLINICA

Via delle Scienze 42,

00100 Roma (RM)

Partita IVA: 01234567890

ESAME RISULTATO UNITA DI MISURA VALORI DI
RIFERIMENTO

Glucosio (Glicemia) 95 mg/dL [70 - 100]
Colesterolo Totale 180 mg/dL <200
Trigliceridi 120 mg/dL <150
Creatinina 0.9 mg/dL [0.7 - 1.2]
Globuli Rossi (RBC) 5.1 milioni/uL [4.5-5.5]
Globuli Bianchi 6.8 mila/uL [4.0-10.0]
(WBC)
Emoglobina (Hgb) 14.5 g/dL [13.0-17.0]





## 2. Agente Analizzatore

In [48]:
from inference_different_data import build_llm

llm = build_llm(0.2)
structured_llm = llm.with_structured_output(DocumentExtraction)

prompt = (
    "Analizza il seguente testo OCR estratto da un PDF.\n\n"
    "TESTO OCR:\n" + raw_text + "\n\n"
    "Il tuo compito:\n"
    "1. Deduci la struttura del documento (NON fare assunzioni sul tipo).\n"
    "2. Per ogni campo, indica nome, tipo, se e' sensibile, e eventuali vincoli.\n"
    "3. Estrai i dati dal documento nel formato dedotto.\n\n"
    "Marca come sensitive: true TUTTI i campi con dati personali."
)

extraction: DocumentExtraction = structured_llm.invoke(prompt)

schema_list = [fd.model_dump() for fd in extraction.schema_def]
print("=== SCHEMA ===")
print(json.dumps(schema_list, indent=2))
print("\n=== DATI SORGENTE ===")
print(json.dumps(extraction.data, indent=2))

=== SCHEMA ===
[
  {
    "field": "laboratorio_nome",
    "type": "string",
    "sensitive": false,
    "constraints": "Nome della struttura sanitaria"
  },
  {
    "field": "paziente_nome_cognome",
    "type": "string",
    "sensitive": true,
    "constraints": "Nome e cognome completi"
  },
  {
    "field": "paziente_codice_fiscale",
    "type": "string",
    "sensitive": true,
    "constraints": "Formato alfanumerico 16 caratteri"
  },
  {
    "field": "paziente_data_nascita",
    "type": "date",
    "sensitive": true,
    "constraints": "Formato GG/MM/AAAA"
  },
  {
    "field": "paziente_indirizzo",
    "type": "string",
    "sensitive": true,
    "constraints": "Indirizzo di residenza"
  },
  {
    "field": "prelievo_data",
    "type": "date",
    "sensitive": false,
    "constraints": "Formato GG/MM/AAAA"
  },
  {
    "field": "medico_richiedente",
    "type": "string",
    "sensitive": false,
    "constraints": "Nome del medico"
  },
  {
    "field": "analisi",
    "type": "arr

## 3. Agente Generatore

In [55]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

N = 1
schema_json = json.dumps(schema_list, indent=2)
source_json = json.dumps(extraction.data, indent=2)

llm2 = build_llm(0.7)
generator = create_agent(
    model=llm2,
    tools=[],
    system_prompt=(
        "Sei un anonimizzatore di dati. Produci ESATTAMENTE " + str(N) + " record. "
        "I campi NON sensibili (quantita', prezzi, totali, IVA, date) "
        "DEVONO rimanere IDENTICI all'originale. "
        "Sostituisci SOLO i campi sensitive (nomi, indirizzi, tax ID, IBAN, codice fiscale di 16 caratteri) con valori fittizi. "
        "NON alterare il numero di items. Output SOLO array JSON."
    ),
)

prompt = (
    "SCHEMA:\n" + schema_json +
    "\n\nDATI ORIGINALI (preserva TUTTI i valori non sensitive):\n" + source_json +
    "\n\nAnonimizza solo i campi sensitive."
)

result = generator.invoke({"messages": [HumanMessage(content=prompt)]})
generated_raw = result["messages"][-1].content
print("Generatore output (primi 500 char):", generated_raw[:500])

Generatore output (primi 500 char): ```json
[
  {
    "laboratorio_nome": "LABORATORIO ANALISI CLINICHE \"VITA NOVA\"",
    "paziente_nome_cognome": "Luca Verdi",
    "paziente_codice_fiscale": "VRLCRA80B02H501X",
    "paziente_data_nascita": "15/03/1980",
    "paziente_indirizzo": "Via delle Mimose 10, 20100 Milano",
    "prelievo_data": "29/05/2026",
    "medico_richiedente": "Dott.ssa Giulia Bianchi",
    "analisi": [
      {
        "esame": "Glucosio (Glicemia)",
        "risultato": 95,
        "unita": "mg/dL",
        "val


## 4. Agente Validatore

In [57]:
llm3 = build_llm(0.1)
structured_val = llm3.with_structured_output(ValidatedRecords)

prompt = (
    "Valida i seguenti record sintetici contro lo schema fornito.\n\n"
    "SCHEMA e constraints:\n" + schema_json +
    "\n\nRECORD DA VALIDARE:\n" + generated_raw +
    "\n\nPer OGNI record: verifica campi, tipi, constraints. "
    "Correggi errori di calcolo. Rimuovi record invalidi."
)

validation = structured_val.invoke(prompt)
validated = validation.records

print(f"Validati: {len(validated)} record")

# Aggiungi questo controllo per evitare l'IndexError
if len(validated) > 0:
    print(json.dumps(validated[0], indent=2)[:600])
else:
    print("Nessun record è sopravvissuto alla validazione. Il modello li ha scartati tutti.")


Validati: 1 record
{
  "status": "INVALIDO",
  "errori": [
    {
      "campo": "paziente_codice_fiscale",
      "errore": "Lunghezza non conforme. Il valore 'VRLCRA80B02H501X' ha 17 caratteri, lo schema ne richiede esattamente 16."
    },
    {
      "campo": "prelievo_data",
      "errore": "Data futura non plausibile. Il valore '29/05/2026' \u00e8 successivo alla data odierna."
    }
  ]
}


## 5. Generazione DINAMICA PDF sintetico
Crea un PDF con la stessa struttura dell'originale ma con dati sintetici.

In [54]:
from fpdf import FPDF
import os

record = validated[0] # Estrae il primo record dall'array validato

pdf = FPDF()
pdf.add_page()
pdf.set_auto_page_break(auto=True, margin=15)

# 1. TITOLO
doc_title = str(record.get("document_type", record.get("doc_type", "SYNTHETIC DOCUMENT"))).upper()
pdf.set_font("Helvetica", "B", 18)
pdf.set_text_color(30, 60, 120)
pdf.cell(0, 12, doc_title, new_x="LMARGIN", new_y="NEXT", align="C")
pdf.ln(4)

pdf.set_draw_color(30, 60, 120)
pdf.set_line_width(0.5)
pdf.line(10, pdf.get_y(), 200, pdf.get_y())
pdf.ln(6)

# Intestazione Sezione Dati
pdf.set_text_color(40, 40, 40)
pdf.set_font("Helvetica", "B", 11)
pdf.cell(0, 8, "DOCUMENT DATA", new_x="LMARGIN", new_y="NEXT")
pdf.set_font("Helvetica", "", 10)

lists_to_render = []

# 2. CICLO DINAMICO DI COPERTURA (Prende qualsiasi chiave generata nel record)
for name, val in record.items():
    if not val or str(val).strip() == "None" or val == [] or val == {}:
        continue
    if name in ["document_type", "doc_type"]: # Già usato nel titolo principale
        continue

    # Se incontra una lista (es. gli esami medici), la salva per appenderla come tabella sotto
    if isinstance(val, list):
        lists_to_render.append((name, val))
        continue

    # Se incontra un sotto-dizionario (es. anagrafica paziente nidificata)
    if isinstance(val, dict):
        pdf.set_font("Helvetica", "B", 10)
        pdf.cell(0, 6, f"{str(name).replace('_', ' ').upper()}:", new_x="LMARGIN", new_y="NEXT")
        pdf.set_font("Helvetica", "", 10)
        for k, v in val.items():
            if v and str(v) != "None":
                pdf.cell(0, 6, f"   - {str(k).replace('_', ' ').title()}: {str(v)[:80]}", new_x="LMARGIN", new_y="NEXT")
    # Se incontra un campo di testo o numero standard
    else:
        formatted_name = str(name).replace('_', ' ').title()
        pdf.cell(0, 6, f"{formatted_name}: {str(val)[:80]}", new_x="LMARGIN", new_y="NEXT")

pdf.ln(6)

# 3. GENERAZIONE TABELLE DINAMICHE
for list_name, list_data in lists_to_render:
    if not list_data or not isinstance(list_data[0], dict):
        continue 
        
    pdf.set_font("Helvetica", "B", 11)
    pdf.cell(0, 8, str(list_name).replace('_', ' ').upper(), new_x="LMARGIN", new_y="NEXT")
    
    # Prende dinamicamente i nomi delle colonne dalle chiavi del primo oggetto (es. "esame", "risultato")
    headers = list(list_data[0].keys())
    num_cols = len(headers)
    col_width = 190 / max(num_cols, 1) 
    
    # Header Tabella
    pdf.set_fill_color(30, 60, 120)
    pdf.set_text_color(255, 255, 255)
    pdf.set_font("Helvetica", "B", 9)
    for h in headers:
        pdf.cell(col_width, 8, str(h).replace('_', ' ').title()[:15], border=1, fill=True, align="C", new_x="RIGHT", new_y="LAST")
    pdf.ln()
    
    # Righe Tabella
    pdf.set_text_color(40, 40, 40)
    pdf.set_font("Helvetica", "", 9)
    for item in list_data:
        for h in headers:
            val = str(item.get(h, ""))[:20]
            pdf.cell(col_width, 7, val, border=1, align="C", new_x="RIGHT", new_y="LAST")
        pdf.ln()
    pdf.ln(6)

# Footer
pdf.ln(10)
pdf.set_font("Helvetica", "I", 8)
pdf.set_text_color(150, 150, 150)
pdf.cell(0, 5, "This is a synthetic document generated for testing purposes.", align="C", new_x="LMARGIN", new_y="NEXT")

os.makedirs("output/notebook_v4", exist_ok=True)
pdf_path = "output/notebook_v4/synthetic_DIFFdata.pdf"
pdf.output(pdf_path)
print(f"PDF generato correttamente nel notebook: {pdf_path}")

PDF generato correttamente nel notebook: output/notebook_v4/synthetic_DIFFdata.pdf


## Bonus: workflow completo in un colpo solo

In [ ]:
from inference_different_data import build_graph

app = build_graph()

result = app.invoke({
    "messages": [
        HumanMessage(content="1"),
        HumanMessage(content="run_id:notebook_v4_full"),
    ],
    "raw_text": raw_text,
    "document_schema": "",
    "source_data": "",
    "generated_data": "",
    "validated_data": "",
    "final_data": [],
    "errors": [],
    "pdf_path": "",
})

print(f"Completato: {len(result['final_data'])} record + PDF: {result['pdf_path']}")

2026-05-28 22:23:26.531 | INFO     | inferencev4:analyze_node:92 - [Agente 1] Analisi del documento - deduzione struttura e estrazione dati
2026-05-28 22:28:29.498 | INFO     | inferencev4:analyze_node:116 - Schema dedotto: 25 campi
2026-05-28 22:28:29.506 | INFO     | inferencev4:generate_node:127 - [Agente 2] Generazione di 1 record sintetici
2026-05-28 22:28:46.706 | INFO     | inferencev4:generate_node:156 - Generati 1 record sintetici
2026-05-28 22:28:46.709 | INFO     | inferencev4:validate_node:161 - [Agente 3] Validazione e correzione dei record generati
2026-05-28 22:29:13.504 | INFO     | inferencev4:validate_node:184 - Validazione completata: 1 record
2026-05-28 22:29:13.507 | INFO     | inferencev4:generate_pdf_node:221 - [Agente 5] Generazione PDF sintetico
2026-05-28 22:29:13.622 | INFO     | inferencev4:generate_pdf_node:383 - PDF sintetico salvato: output/notebook_v4_full/synthetic_invoice.pdf
2026-05-28 22:29:13.625 | INFO     | inferencev4:save_node:389 - [Salvataggio

Completato: 1 record + PDF: output/notebook_v4_full/synthetic_invoice.pdf
